# 04 — Финальное решение V26

In [1]:
from __future__ import annotations

import gc
import json
import math
import shutil
import subprocess
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import polars as pl
from catboost import CatBoostClassifier, CatBoostRegressor

SEED = 42
EXPECTED_USERS = 250_000
HORIZON = 30
DEVICES = "0"

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("NVIDIA GPU not detected")

if subprocess.run(
    ["nvidia-smi", "-L"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=False,
).returncode != 0:
    raise RuntimeError("NVIDIA GPU not detected")

GPU_NAME = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip().splitlines()[0]

print("GPU:", GPU_NAME)


GPU: NVIDIA GeForce RTX 4070 SUPER


In [2]:
V9_CANONICAL_POOL = [
    date(2025, 2, 13),
    date(2025, 3, 26),
    date(2025, 5, 7),
    date(2025, 6, 18),
    date(2025, 7, 30),
    date(2025, 9, 10),
    date(2025, 10, 22),
    date(2025, 11, 12),
    date(2025, 12, 3),
    date(2025, 12, 15),
    date(2026, 1, 14),
]

FIXED_RECENT_WEIGHT = 0.46619
FIXED_HURDLE_WEIGHT = 0.53381
CONTROL_GAMMA = 2.0
LOW_VALUE_BLOCK_UNTIL_RANK = 0.60
LOW_VALUE_FULL_V12_FROM_RANK = 0.90
PURCHASE_AUX_SEED = 1343

A_CFG = {
    "iterations": 1784,
    "depth": 8,
    "learning_rate": 0.02020453138039556,
    "l2_leaf_reg": 3.7296561917972872,
    "random_strength": 0.11299833172000943,
    "bootstrap_type": "No",
    "border_count": 254,
    "random_seed": 1043,
    "loss_kind": "RMSE",
    "grow_policy": "SymmetricTree",
    "score_function": "NewtonL2",
    "leaf_estimation_method": "Gradient",
    "leaf_estimation_iterations": 3,
    "boosting_type": "Plain",
    "decay_days": 180.0,
}

B_CFG = {
    "iterations": 1214,
    "depth": 7,
    "learning_rate": 0.02976696718584,
    "l2_leaf_reg": 6.987063179169868,
    "random_strength": 0.0208095549419522,
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 0.8512703294400126,
    "border_count": 254,
    "random_seed": 1143,
    "loss_kind": "RMSE",
    "grow_policy": "SymmetricTree",
    "score_function": "NewtonL2",
    "leaf_estimation_method": "Gradient",
    "leaf_estimation_iterations": 3,
    "boosting_type": "Plain",
    "decay_days": 180.0,
}

POS_CFG = {
    "iterations": 662,
    "depth": 7,
    "learning_rate": 0.0393118775835085,
    "l2_leaf_reg": 10.954452809245176,
    "random_strength": 0.0457125748507678,
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 0.3799888701178133,
    "border_count": 254,
    "random_seed": 343,
    "loss_kind": "RMSE",
    "grow_policy": "SymmetricTree",
    "score_function": "Cosine",
    "leaf_estimation_method": "Gradient",
    "leaf_estimation_iterations": 1,
    "boosting_type": "Plain",
    "decay_days": 365.0,
}

AUX_CFG = {
    "iterations": 327,
    "depth": 6,
    "learning_rate": 0.0472132001034989,
    "l2_leaf_reg": 5.077956932853544,
    "random_strength": 0.1288362327218055,
    "bootstrap_type": "Bayesian",
    "bagging_temperature": 0.1779700675979256,
    "loss_kind": "RMSE",
    "decay_days": 365.0,
}

A_REPR_TOP20 = [
    "repr_dense_10", "repr_dense_15", "repr_dense_09", "repr_dense_02",
    "repr_dense_12", "repr_dense_07", "repr_dense_13", "repr_dense_17",
    "repr_dense_06", "repr_dense_00", "repr_dense_03", "repr_dense_11",
    "repr_dense_05", "repr_dense_23", "repr_dense_04", "repr_dense_14",
    "repr_dense_21", "repr_dense_16", "repr_dense_19", "repr_dense_18",
]

B_REPR_TOP20 = [
    "repr_dense_03", "repr_dense_16", "repr_dense_17", "repr_dense_13",
    "repr_dense_12", "repr_dense_23", "repr_dense_00", "repr_dense_04",
    "repr_dense_11", "repr_dense_07", "repr_dense_15", "repr_dense_19",
    "repr_dense_14", "repr_dense_20", "repr_dense_08", "repr_dense_22",
    "repr_dense_10", "repr_dense_18", "repr_dense_02", "repr_dense_05",
]

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
PREP_DIR = PROJECT_DIR / "artifacts" / "prepare_features"
BASE_DIR = PROJECT_DIR / "artifacts" / "base_model"
REPR_DIR = PROJECT_DIR / "artifacts" / "neural_features" / "representation"
ARTIFACT_ROOT = PROJECT_DIR / "artifacts" / "solution_v26"
CACHE_DIR = ARTIFACT_ROOT / "_cache"
HYBRID_DIR = CACHE_DIR / "hybrid"
AUX_DIR = CACHE_DIR / "aux05"
TARGET_DIR = CACHE_DIR / "targets"

for p in [ARTIFACT_ROOT, CACHE_DIR, HYBRID_DIR, AUX_DIR, TARGET_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def first_existing(*paths: Path) -> Path:
    for path in paths:
        if path.is_file():
            return path
    raise FileNotFoundError("\n".join(str(p) for p in paths))

TRAIN_PATH = first_existing(DATA_DIR / "train.parquet", PROJECT_DIR / "train.parquet")
SAMPLE_PATH = first_existing(
    DATA_DIR / "sample_submit.csv",
    DATA_DIR / "sample_submission.csv",
    PROJECT_DIR / "sample_submit.csv",
    PROJECT_DIR / "sample_submission.csv",
)
TEMP_HIST_PATH = first_existing(PREP_DIR / "temporal_features_historical_K5.parquet")
BASE_FINAL_PATH = first_existing(BASE_DIR / "final_candidate_predictions.parquet")
BASE_FRAME_DIR = BASE_DIR / "_cache" / "model_frames"
PURCHASE_SEQ_PATH = BASE_DIR / "_cache" / "purchase_aux" / "purchase_sequence_arrays_v2_next_purchase_nohistory_neutral.npz"

data = pl.read_parquet(TRAIN_PATH)
if data.schema["event_date"] != pl.Date:
    data = data.with_columns(pl.col("event_date").cast(pl.Date))

submit_template = pl.read_csv(SAMPLE_PATH).select("user_id")
all_users = submit_template.unique(maintain_order=True)

assert all_users.height == EXPECTED_USERS
assert all_users["user_id"].n_unique() == EXPECTED_USERS

max_date = data["event_date"].max()
assert max_date == date(2026, 2, 13)

for anchor in V9_CANONICAL_POOL:
    p = BASE_FRAME_DIR / f"model_frame_{anchor}_target1.parquet"
    if not p.is_file():
        raise FileNotFoundError(f"Run 02_base_model.ipynb first: {p}")

final_base_cache = BASE_FRAME_DIR / f"model_frame_{max_date}_target0.parquet"
if not final_base_cache.is_file():
    raise FileNotFoundError(f"Run 02_base_model.ipynb first: {final_base_cache}")

for anchor in [*V9_CANONICAL_POOL, max_date]:
    p = REPR_DIR / f"repr_dense_{anchor}.npz"
    if not p.is_file():
        raise FileNotFoundError(f"Run 03_neural_features.ipynb first: {p}")

user_ids_path = REPR_DIR / "user_ids.npy"
if not user_ids_path.is_file():
    raise FileNotFoundError(user_ids_path)

repr_user_ids = np.load(user_ids_path)
assert np.array_equal(
    repr_user_ids.astype(np.int64),
    all_users["user_id"].cast(pl.Int64).to_numpy(),
)

hist_schema = pl.read_parquet_schema(TEMP_HIST_PATH)

v4_meta = [
    "future_arch_availability_score",
    "future_arch_measurement_confidence",
    "future_arch_quality_score",
    "future_arch5_dominant_weight",
    "future_arch5_entropy",
    "future_arch5_effective_count",
    "current_arch5_weight_eligible",
    "current_arch5_measurement_confidence",
    "current_arch5_dominant_weight",
    "current_arch5_entropy",
    "current_arch5_effective_count",
    "future_vs_current_arch5_cosine",
    "future_vs_current_arch5_l1",
    "future_vs_current_arch5_dominant_same",
]
v4_meta = [c for c in v4_meta if c in hist_schema]

V4_FEATURE_COLS = []
for c in hist_schema:
    keep = (
        c.startswith("aux_mean_")
        or c.startswith("future_arch_eb__")
        or c.startswith("future_arch_rank__")
        or c.startswith("future_arch5_w_")
        or c.startswith("current_arch5_w_")
        or c.startswith("future_arch5_delta_w_")
        or c in v4_meta
    )
    if keep and c not in V4_FEATURE_COLS:
        V4_FEATURE_COLS.append(c)

probe_schema = pl.read_parquet_schema(
    BASE_FRAME_DIR / "model_frame_2026-01-14_target1.parquet"
)
exclude = {
    "user_id", "anchor_date", "target",
    "aux_fit_cutoff", "arch_fit_cutoff",
    *V4_FEATURE_COLS,
}
BASE_FEATURE_COLS = [c for c in probe_schema if c not in exclude]
BASE_VALUE_FEATURE_COLS = list(dict.fromkeys(BASE_FEATURE_COLS + V4_FEATURE_COLS))

assert len(BASE_FEATURE_COLS) == 591
assert len(BASE_VALUE_FEATURE_COLS) == 662

print("Base features:", len(BASE_FEATURE_COLS))
print("Value features:", len(BASE_VALUE_FEATURE_COLS))


Base features: 591
Value features: 662


## Данные

In [3]:
def recency_weights(frame: pl.DataFrame, decay_days: float) -> np.ndarray:
    anchor_days = frame["anchor_date"].cast(pl.Int32).to_numpy()
    newest = int(np.max(anchor_days))
    age = newest - anchor_days
    w = np.exp(-age / decay_days).astype(np.float32)
    return w / max(float(w.mean()), 1e-12)

def _matrix(
    frame: pl.DataFrame,
    cols: list[str],
    mask: np.ndarray | None = None,
) -> np.ndarray:
    part = frame
    if mask is not None:
        part = frame.filter(pl.Series("__mask", np.asarray(mask, dtype=bool)))
    return part.select(cols).to_numpy().astype(np.float32, copy=False)

def model_params(cfg: dict, classifier: bool = False) -> dict:
    p = {
        "iterations": int(cfg["iterations"]),
        "loss_function": "Logloss" if classifier else "RMSE",
        "eval_metric": "Logloss" if classifier else "RMSE",
        "depth": int(cfg["depth"]),
        "learning_rate": float(cfg["learning_rate"]),
        "l2_leaf_reg": float(cfg["l2_leaf_reg"]),
        "random_strength": float(cfg["random_strength"]),
        "grow_policy": cfg.get("grow_policy", "SymmetricTree"),
        "score_function": cfg.get("score_function", "Cosine"),
        "border_count": int(cfg.get("border_count", 254)),
        "boosting_type": cfg.get("boosting_type", "Plain"),
        "random_seed": int(cfg["random_seed"]),
        "allow_writing_files": False,
        "task_type": "GPU",
        "devices": DEVICES,
        "verbose": False,
        "bootstrap_type": cfg.get("bootstrap_type", "Bayesian"),
    }

    p["leaf_estimation_method"] = cfg.get(
        "leaf_estimation_method",
        "Newton" if classifier else "Gradient",
    )
    p["leaf_estimation_iterations"] = int(
        cfg.get("leaf_estimation_iterations", 10 if classifier else 1)
    )

    if p["bootstrap_type"] == "Bayesian":
        p["bagging_temperature"] = float(cfg.get("bagging_temperature", 0.6))
    elif p["bootstrap_type"] in {"Bernoulli", "Poisson"}:
        p["subsample"] = float(cfg.get("subsample", 0.8))

    return p

def future_components(anchor: date) -> pl.DataFrame:
    cp = TARGET_DIR / f"future_components_{anchor}.parquet"
    if cp.is_file():
        return pl.read_parquet(cp)

    start = anchor + timedelta(days=1)
    end = anchor + timedelta(days=HORIZON)

    agg = (
        data.filter(pl.col("event_date").is_between(start, end, closed="both"))
        .group_by("user_id")
        .agg([
            pl.col("gmv").sum().alias("__target_check_raw"),
            (pl.col("gmv") > 0).sum().alias("future_spend_days"),
        ])
    )

    out = (
        all_users.join(agg, on="user_id", how="left")
        .with_columns([
            pl.col("__target_check_raw")
            .fill_null(0)
            .clip(lower_bound=0)
            .cast(pl.Float32),
            pl.col("future_spend_days")
            .fill_null(0)
            .cast(pl.Float32),
        ])
    )

    out.write_parquet(cp, compression="zstd")
    return out

def load_base_frame(anchor: date, with_target: bool) -> pl.DataFrame:
    cp = BASE_FRAME_DIR / f"model_frame_{anchor}_target{int(with_target)}.parquet"
    frame = pl.read_parquet(cp)

    if frame.schema["anchor_date"] != pl.Date:
        frame = frame.with_columns(pl.col("anchor_date").cast(pl.Date))

    assert frame.height == EXPECTED_USERS
    assert frame["user_id"].n_unique() == EXPECTED_USERS

    if not with_target:
        return frame

    comp = future_components(anchor)
    out = (
        frame.with_row_index("__row")
        .join(comp, on="user_id", how="left", validate="m:1")
        .sort("__row")
        .drop("__row")
    )

    diff = np.abs(
        out["target"].to_numpy().astype(np.float64)
        - out["__target_check_raw"].to_numpy().astype(np.float64)
    )
    assert float(diff.max()) < 1e-3
    return out


In [4]:
PURCHASE_STATE_FEATURES = [
    "purchase_has_history",
    "purchase_log_count",
    "purchase_last_log_gmv",
    "purchase_mean3_log_gmv",
    "purchase_mean5_log_gmv",
    "purchase_mean10_log_gmv",
    "purchase_std5_log_gmv",
    "purchase_std10_log_gmv",
    "purchase_last_log_ticket",
    "purchase_mean5_log_ticket",
    "purchase_last_gap_days",
    "purchase_mean3_gap_days",
]

PURCHASE_AUX_FEATURE_COLS = [
    "purchase_aux_pred_next_log_gmv",
    "purchase_aux_hist_std5_log_gmv",
    "purchase_aux_has_history",
]

def rolling_mean(x: np.ndarray, window: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    cs = np.concatenate(([0.0], np.cumsum(x)))
    j = np.arange(n)
    s = np.maximum(0, j - window + 1)
    return ((cs[j + 1] - cs[s]) / (j - s + 1)).astype(np.float32)

def rolling_std(x: np.ndarray, window: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    cs = np.concatenate(([0.0], np.cumsum(x)))
    cs2 = np.concatenate(([0.0], np.cumsum(x * x)))
    j = np.arange(n)
    s = np.maximum(0, j - window + 1)
    cnt = (j - s + 1).astype(np.float64)
    mean = (cs[j + 1] - cs[s]) / cnt
    mean2 = (cs2[j + 1] - cs2[s]) / cnt
    return np.sqrt(np.maximum(mean2 - mean * mean, 0.0)).astype(np.float32)

def build_purchase_sequence() -> dict[str, np.ndarray]:
    if PURCHASE_SEQ_PATH.is_file():
        z = np.load(PURCHASE_SEQ_PATH)
        return {
            "uid": z["uid"],
            "day": z["day"],
            "state": z["state"],
            "transition_x": z["transition_x"],
            "transition_y": z["transition_y"],
            "transition_label_day": z["transition_label_day"],
            "group_starts": z["group_starts"],
            "group_ends": z["group_ends"],
        }

    p = (
        data.filter(pl.col("gmv") > 0)
        .select([
            "user_id",
            "event_date",
            pl.col("gmv").cast(pl.Float64),
            pl.col("to_ord").cast(pl.Float64),
        ])
        .sort(["user_id", "event_date"])
    )

    uid = p["user_id"].to_numpy()
    day = p["event_date"].cast(pl.Int32).to_numpy().astype(np.int32)
    gmv = p["gmv"].to_numpy().astype(np.float64)
    orders = p["to_ord"].to_numpy().astype(np.float64)
    log_gmv = np.log1p(np.clip(gmv, 0, None))
    log_ticket = np.log1p(
        np.clip(gmv / np.maximum(orders, 1.0), 0, None)
    )

    state = np.zeros(
        (len(uid), len(PURCHASE_STATE_FEATURES)),
        dtype=np.float32,
    )

    starts = np.r_[
        0,
        np.flatnonzero(uid[1:] != uid[:-1]) + 1,
    ].astype(np.int64)
    ends = np.r_[starts[1:], len(uid)].astype(np.int64)

    for s, e in zip(starts, ends):
        x = log_gmv[s:e]
        t = log_ticket[s:e]
        d = day[s:e].astype(np.float64)
        m = e - s

        state[s:e, 0] = 1.0
        state[s:e, 1] = np.log1p(np.arange(1, m + 1))
        state[s:e, 2] = x
        state[s:e, 3] = rolling_mean(x, 3)
        state[s:e, 4] = rolling_mean(x, 5)
        state[s:e, 5] = rolling_mean(x, 10)
        state[s:e, 6] = rolling_std(x, 5)
        state[s:e, 7] = rolling_std(x, 10)
        state[s:e, 8] = t
        state[s:e, 9] = rolling_mean(t, 5)

        gaps = np.full(m, 999.0, dtype=np.float64)
        if m > 1:
            gaps[1:] = np.diff(d)
        state[s:e, 10] = gaps

        gap_mean = np.full(m, 999.0, dtype=np.float32)
        if m > 1:
            gap_mean[1:] = rolling_mean(gaps[1:], 3)
        state[s:e, 11] = gap_mean

    has_next = uid[:-1] == uid[1:]
    transition_idx = np.flatnonzero(has_next).astype(np.int64)
    label_idx = transition_idx + 1

    return {
        "uid": uid,
        "day": day,
        "state": state,
        "transition_x": state[transition_idx].astype(np.float32, copy=False),
        "transition_y": log_gmv[label_idx].astype(np.float32, copy=False),
        "transition_label_day": day[label_idx].astype(np.int32, copy=False),
        "group_starts": starts,
        "group_ends": ends,
    }

PURCHASE_SEQ = build_purchase_sequence()

def purchase_state(anchor: date) -> pl.DataFrame:
    cp = AUX_DIR / f"state_{anchor}.parquet"
    if cp.is_file():
        return pl.read_parquet(cp)

    anchor_day = (anchor - date(1970, 1, 1)).days
    seq = PURCHASE_SEQ
    idxs = []

    for s, e in zip(seq["group_starts"], seq["group_ends"]):
        local = np.searchsorted(
            seq["day"][s:e],
            anchor_day,
            side="right",
        ) - 1
        if local >= 0:
            idxs.append(int(s + local))

    idxs = np.asarray(idxs, dtype=np.int64)

    state_df = pl.DataFrame({
        "user_id": seq["uid"][idxs],
        **{
            c: seq["state"][idxs, j]
            for j, c in enumerate(PURCHASE_STATE_FEATURES)
        },
    })

    out = (
        all_users.with_row_index("__row")
        .join(state_df, on="user_id", how="left", validate="1:1")
        .sort("__row")
        .drop("__row")
    )

    exprs = []
    for c in PURCHASE_STATE_FEATURES:
        default = 999.0 if c in {
            "purchase_last_gap_days",
            "purchase_mean3_gap_days",
        } else 0.0
        exprs.append(
            pl.col(c).fill_null(default).cast(pl.Float32)
        )

    out = out.with_columns(exprs)
    out.write_parquet(cp, compression="zstd")
    return out

def aux_params() -> dict:
    return {
        "iterations": int(AUX_CFG["iterations"]),
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "depth": int(AUX_CFG["depth"]),
        "learning_rate": float(AUX_CFG["learning_rate"]),
        "l2_leaf_reg": float(AUX_CFG["l2_leaf_reg"]),
        "random_strength": float(AUX_CFG["random_strength"]),
        "border_count": 254,
        "random_seed": PURCHASE_AUX_SEED,
        "allow_writing_files": False,
        "task_type": "GPU",
        "devices": DEVICES,
        "verbose": False,
        "bootstrap_type": "Bayesian",
        "bagging_temperature": float(AUX_CFG["bagging_temperature"]),
    }

def selected_purchase_aux(anchor: date) -> pl.DataFrame:
    cp = AUX_DIR / f"aux05_{anchor}.parquet"
    if cp.is_file():
        return pl.read_parquet(cp)

    seq = PURCHASE_SEQ
    anchor_day = (anchor - date(1970, 1, 1)).days
    mask = seq["transition_label_day"] <= anchor_day

    Xtr = seq["transition_x"][mask].astype(np.float32, copy=False)
    ytr = seq["transition_y"][mask].astype(np.float32, copy=False)
    label_day = seq["transition_label_day"][mask].astype(np.int32, copy=False)

    sw = np.exp(
        -(anchor_day - label_day) / float(AUX_CFG["decay_days"])
    ).astype(np.float64)
    sw /= max(float(sw.mean()), 1e-12)

    state = purchase_state(anchor)
    Xva = state.select(PURCHASE_STATE_FEATURES).to_numpy().astype(
        np.float32,
        copy=False,
    )

    model = CatBoostRegressor(**aux_params())
    model.fit(Xtr, ytr, sample_weight=sw, verbose=False)
    pred = model.predict(Xva).astype(np.float64)

    assert np.isfinite(pred).all()

    out = pl.DataFrame({
        "user_id": state["user_id"],
        "purchase_aux_pred_next_log_gmv": pred.astype(np.float32),
        "purchase_aux_hist_std5_log_gmv": state["purchase_std5_log_gmv"],
        "purchase_aux_has_history": state["purchase_has_history"],
    })
    out.write_parquet(cp, compression="zstd")

    del model, Xtr, ytr, Xva, sw, state
    gc.collect()
    return out


In [5]:
REPR_COLS = [f"repr_dense_{j:02d}" for j in range(24)]

def load_repr(anchor: date) -> pl.DataFrame:
    z = np.load(REPR_DIR / f"repr_dense_{anchor}.npz")
    q = z["q"].astype(np.float32)
    scale = z["scale"].astype(np.float32)

    assert q.shape == (EXPECTED_USERS, 24)
    assert scale.shape == (24,)

    emb = q * scale[None, :]
    assert np.isfinite(emb).all()

    payload = {"user_id": all_users["user_id"]}
    for j, c in enumerate(REPR_COLS):
        payload[c] = emb[:, j]

    return pl.DataFrame(payload)

def hybrid_frame(anchor: date, with_target: bool) -> pl.DataFrame:
    cp = HYBRID_DIR / f"hybrid_{anchor}_{int(with_target)}.parquet"
    if cp.is_file():
        return pl.read_parquet(cp)

    frame = load_base_frame(anchor, with_target)
    aux = selected_purchase_aux(anchor)
    rep = load_repr(anchor)

    out = (
        frame.with_row_index("__row")
        .join(aux, on="user_id", how="left", validate="m:1")
        .join(rep, on="user_id", how="left", validate="m:1")
        .sort("__row")
        .drop("__row")
    )

    assert out.height == EXPECTED_USERS

    for c in [*PURCHASE_AUX_FEATURE_COLS, *REPR_COLS]:
        assert out[c].null_count() == 0
        assert np.isfinite(out[c].to_numpy()).all()

    out.write_parquet(cp, compression="zstd")
    return out

A_FEATURE_COLS = BASE_VALUE_FEATURE_COLS + A_REPR_TOP20
B_FEATURE_COLS = (
    BASE_VALUE_FEATURE_COLS
    + PURCHASE_AUX_FEATURE_COLS
    + B_REPR_TOP20
)

assert len(A_FEATURE_COLS) == 682
assert len(B_FEATURE_COLS) == 685


## Модели

In [6]:
train_frames = []

for i, anchor in enumerate(V9_CANONICAL_POOL, 1):
    print(f"[{i}/{len(V9_CANONICAL_POOL)}] {anchor}")
    train_frames.append(hybrid_frame(anchor, True))

full = pl.concat(
    train_frames,
    how="vertical_relaxed",
    rechunk=False,
)
del train_frames
gc.collect()

final = hybrid_frame(max_date, False)

y = full["target"].to_numpy().astype(np.float64)
days = full["future_spend_days"].to_numpy().astype(np.float64)
pos = y > 0

assert np.all(days[pos] >= 1.0)

a_target = np.log(days[pos])
b_target = np.log1p(y[pos]) - a_target

sw_a = recency_weights(full, A_CFG["decay_days"])[pos]
sw_b = recency_weights(full, B_CFG["decay_days"])[pos]

Xtr = _matrix(full, A_FEATURE_COLS, pos)
Xva = _matrix(final, A_FEATURE_COLS)

model_a = CatBoostRegressor(**model_params(A_CFG))
model_a.fit(Xtr, a_target, sample_weight=sw_a, verbose=False)
pred_a = model_a.predict(Xva).astype(np.float64)

del model_a, Xtr, Xva, sw_a
gc.collect()

Xtr = _matrix(full, B_FEATURE_COLS, pos)
Xva = _matrix(final, B_FEATURE_COLS)

model_b = CatBoostRegressor(**model_params(B_CFG))
model_b.fit(Xtr, b_target, sample_weight=sw_b, verbose=False)
pred_b = model_b.predict(Xva).astype(np.float64)

del model_b, Xtr, Xva, sw_b
gc.collect()

assert np.isfinite(pred_a).all()
assert np.isfinite(pred_b).all()


[1/11] 2025-02-13
[2/11] 2025-03-26
[3/11] 2025-05-07
[4/11] 2025-06-18
[5/11] 2025-07-30
[6/11] 2025-09-10
[7/11] 2025-10-22
[8/11] 2025-11-12
[9/11] 2025-12-03
[10/11] 2025-12-15
[11/11] 2026-01-14


In [7]:
base_train = pl.concat(
    [
        load_base_frame(anchor, True)
        for anchor in V9_CANONICAL_POOL
    ],
    how="vertical_relaxed",
    rechunk=False,
)
base_final = load_base_frame(max_date, False)

y_pos = base_train["target"].to_numpy().astype(np.float64)
mask_pos = y_pos > 0

Xtr = _matrix(base_train, BASE_FEATURE_COLS, mask_pos)
Xva = _matrix(base_final, BASE_FEATURE_COLS)
sw = recency_weights(base_train, POS_CFG["decay_days"])[mask_pos]

positive_model = CatBoostRegressor(**model_params(POS_CFG))
positive_model.fit(
    Xtr,
    np.log1p(y_pos[mask_pos]),
    sample_weight=sw,
    verbose=False,
)
pred_positive = np.clip(
    positive_model.predict(Xva).astype(np.float64),
    0.0,
    None,
)

del positive_model, Xtr, Xva, sw, base_train, base_final
gc.collect()

assert np.isfinite(pred_positive).all()


## Сабмит

In [8]:
def prediction_percentile_rank(values: np.ndarray) -> np.ndarray:
    v = np.asarray(values, dtype=np.float64)
    sorted_v = np.sort(v)
    rank = np.searchsorted(
        sorted_v,
        v,
        side="right",
    ).astype(np.float64) / len(v)
    return np.clip(rank, 0.0, 1.0)

def low_value_rank_gate(base_pred_log: np.ndarray) -> np.ndarray:
    rank = prediction_percentile_rank(base_pred_log)
    x = (
        rank - LOW_VALUE_BLOCK_UNTIL_RANK
    ) / (
        LOW_VALUE_FULL_V12_FROM_RANK
        - LOW_VALUE_BLOCK_UNTIL_RANK
    )
    x = np.clip(x, 0.0, 1.0)
    return np.clip(
        x * x * (3.0 - 2.0 * x),
        0.0,
        1.0,
    )

def reconstruct_v9_positive(
    recent: np.ndarray,
    occurrence: np.ndarray,
    base: np.ndarray,
) -> np.ndarray:
    recent = np.asarray(recent, dtype=np.float64)
    occurrence = np.clip(
        np.asarray(occurrence, dtype=np.float64),
        0.0,
        1.0,
    )
    base = np.clip(
        np.asarray(base, dtype=np.float64),
        0.0,
        None,
    )

    hurdle = np.clip(
        (
            base
            - FIXED_RECENT_WEIGHT * recent
        )
        / FIXED_HURDLE_WEIGHT,
        0.0,
        None,
    )

    return np.clip(
        hurdle / np.maximum(occurrence, 1e-12),
        0.0,
        None,
    )

def compose_v26(
    recent: np.ndarray,
    occurrence: np.ndarray,
    positive: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
) -> np.ndarray:
    recent = np.asarray(recent, dtype=np.float64)
    occurrence = np.clip(
        np.asarray(occurrence, dtype=np.float64),
        0.0,
        1.0,
    )
    positive = np.clip(
        np.asarray(positive, dtype=np.float64),
        0.0,
        None,
    )

    base = np.clip(
        FIXED_RECENT_WEIGHT * recent
        + FIXED_HURDLE_WEIGHT * occurrence * positive,
        0.0,
        None,
    )

    ab_severity = np.clip(
        np.asarray(pred_a, dtype=np.float64)
        + np.asarray(pred_b, dtype=np.float64),
        0.0,
        None,
    )

    full = np.clip(
        FIXED_RECENT_WEIGHT * recent
        + FIXED_HURDLE_WEIGHT * occurrence * ab_severity,
        0.0,
        None,
    )

    delta = full - base
    up_gate = (
        occurrence ** CONTROL_GAMMA
        * low_value_rank_gate(base)
    )
    safe_delta = np.where(
        delta > 0.0,
        up_gate * delta,
        delta,
    )

    pred = np.clip(
        base + safe_delta,
        0.0,
        None,
    )
    assert np.isfinite(pred).all()
    return pred

base_final_pred = (
    submit_template.with_row_index("__row")
    .join(
        pl.read_parquet(BASE_FINAL_PATH),
        on="user_id",
        how="left",
        validate="1:1",
    )
    .sort("__row")
    .drop("__row")
)

recent = base_final_pred["v9_recent_log"].to_numpy().astype(np.float64)
occurrence = base_final_pred["v9_occurrence"].to_numpy().astype(np.float64)
base_log = base_final_pred["v9_base_log"].to_numpy().astype(np.float64)
positive_exact = reconstruct_v9_positive(
    recent,
    occurrence,
    base_log,
)

pred_log = compose_v26(
    recent,
    occurrence,
    pred_positive,
    pred_a,
    pred_b,
)

components = pl.DataFrame({
    "user_id": submit_template["user_id"],
    "recent_exact": recent,
    "occurrence_exact": occurrence,
    "positive_exact": positive_exact,
    "positive_pos09": pred_positive,
    "pred_a_j065": pred_a,
    "pred_b_j065": pred_b,
    "v9_base_log": base_log,
    "v26_pred_log": pred_log,
})

COMPONENTS_PATH = ARTIFACT_ROOT / "final_components.parquet"
components.write_parquet(COMPONENTS_PATH, compression="zstd")

submission = pl.DataFrame({
    "user_id": submit_template["user_id"],
    "predict": np.clip(
        np.expm1(np.clip(pred_log, 0.0, 30.0)),
        0.0,
        None,
    ),
})

SUBMISSION_PATH = ARTIFACT_ROOT / "submission_v26_J065_e96d822fa5.csv"
submission.write_csv(SUBMISSION_PATH)

contract = {
    "solution": "J065_e96d822fa5",
    "A": "A_REPRK20__2ffcea60a1",
    "B": "B_REPRK20__31a21f6262",
    "positive": "POS_09__59b5afea6e",
    "occurrence": "OCCURRENCE_EXACT",
    "recent": "RECENT_EXACT",
    "aux": "AUX_05__89267da83d",
    "A_feature_cols": A_FEATURE_COLS,
    "B_feature_cols": B_FEATURE_COLS,
    "base_feature_cols": BASE_FEATURE_COLS,
    "base_value_feature_cols": BASE_VALUE_FEATURE_COLS,
}
(ARTIFACT_ROOT / "feature_contract.json").write_text(
    json.dumps(contract, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

assert submission.height == EXPECTED_USERS
assert submission["predict"].null_count() == 0
assert np.isfinite(submission["predict"].to_numpy()).all()
assert float(submission["predict"].min()) >= 0.0

print("Components:", COMPONENTS_PATH.resolve())
print("Submission:", SUBMISSION_PATH.resolve())


Components: D:\baboteek_holmotech\artifacts\solution_v26\final_components.parquet
Submission: D:\baboteek_holmotech\artifacts\solution_v26\submission_v26_J065_e96d822fa5.csv
